In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, random_split
import json
from PIL import Image

In [2]:
# 检查设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
# 数据预处理
transform = transforms.Compose([
    transforms.Resize((350, 1600)),  # 保持原始宽高比
    transforms.CenterCrop((320, 1536)),  # 中心裁剪以去除边缘噪点
    transforms.Resize((80, 384)),  # 按比例缩小到1/4
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [4]:
# 加载数据集
dataset = datasets.ImageFolder(r'D:\STUDY\internship\dataset\images', transform=transform)

# 保存类别映射
with open('class_indices.json', 'w') as f:
    json.dump(dataset.class_to_idx, f)

# 划分训练集和验证集
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# 创建数据加载器
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4)  # 减小批大小
val_loader = DataLoader(val_dataset, batch_size=16, num_workers=4)

In [5]:
# 针对1600x350特殊尺寸的CNN模型
class AspectAwareCNN(nn.Module):
    def __init__(self, num_classes=5):
        super(AspectAwareCNN, self).__init__()
        # 输入尺寸: [3, 80, 384]
        self.conv_layers = nn.Sequential(
            # 第1层卷积
            nn.Conv2d(3, 32, kernel_size=(3, 7), padding=(1, 3)),  # 使用宽核适应宽图像
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d((2, 4)),  # 输出: [32, 40, 96]
            
            # 第2层卷积
            nn.Conv2d(32, 64, kernel_size=(3, 7), padding=(1, 3)),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d((2, 4)),  # 输出: [64, 20, 24]
            
            # 第3层卷积
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d((2, 2)),  # 输出: [128, 10, 12]
        )
        
        # 自适应池化适应不同尺寸
        self.adaptive_pool = nn.AdaptiveAvgPool2d((5, 6))
        
        self.fc_layers = nn.Sequential(
            nn.Linear(128 * 5 * 6, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        x = self.conv_layers(x)
        x = self.adaptive_pool(x)
        x = torch.flatten(x, 1)
        return self.fc_layers(x)

In [6]:
# 初始化模型
model = AspectAwareCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)  # 降低学习率

In [7]:
# 训练函数
def train(model, loader, optimizer, criterion):
    model.train()
    running_loss = 0.0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
    return running_loss / len(loader.dataset)

In [8]:
# 验证函数
def validate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    
    return running_loss / total, correct / total

In [9]:
# 训练循环
num_epochs = 30
best_acc = 0.0

for epoch in range(num_epochs):
    train_loss = train(model, train_loader, optimizer, criterion)
    val_loss, val_acc = validate(model, val_loader, criterion)
    
    # 保存最佳模型
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')
    
    print(f"Epoch {epoch+1}/{num_epochs} | "
          f"Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | "
          f"Val Acc: {val_acc:.4f} | "
          f"Best Acc: {best_acc:.4f}")

print("训练完成，最佳模型已保存!")

Epoch 1/30 | Train Loss: 1.0688 | Val Loss: 0.7702 | Val Acc: 0.6700 | Best Acc: 0.6700
Epoch 2/30 | Train Loss: 0.6726 | Val Loss: 0.4928 | Val Acc: 0.8100 | Best Acc: 0.8100
Epoch 3/30 | Train Loss: 0.5635 | Val Loss: 0.3562 | Val Acc: 0.8800 | Best Acc: 0.8800
Epoch 4/30 | Train Loss: 0.4951 | Val Loss: 0.4214 | Val Acc: 0.8200 | Best Acc: 0.8800
Epoch 5/30 | Train Loss: 0.4756 | Val Loss: 0.4254 | Val Acc: 0.7950 | Best Acc: 0.8800
Epoch 6/30 | Train Loss: 0.4943 | Val Loss: 0.2954 | Val Acc: 0.8650 | Best Acc: 0.8800
Epoch 7/30 | Train Loss: 0.4215 | Val Loss: 0.2982 | Val Acc: 0.8500 | Best Acc: 0.8800
Epoch 8/30 | Train Loss: 0.3978 | Val Loss: 0.5957 | Val Acc: 0.7500 | Best Acc: 0.8800
Epoch 9/30 | Train Loss: 0.4511 | Val Loss: 0.2739 | Val Acc: 0.9000 | Best Acc: 0.9000
Epoch 10/30 | Train Loss: 0.4178 | Val Loss: 0.3406 | Val Acc: 0.8250 | Best Acc: 0.9000
Epoch 11/30 | Train Loss: 0.3499 | Val Loss: 0.2993 | Val Acc: 0.8600 | Best Acc: 0.9000
Epoch 12/30 | Train Loss: 0.37

In [10]:
# 预测函数示例
def predict_image(image_path):
    # 加载类别映射
    with open('class_indices.json') as f:
        class_mapping = json.load(f)
    idx_to_class = {v: k for k, v in class_mapping.items()}
    
    # 加载并预处理图像
    image = Image.open(image_path).convert('RGB')
    image_tensor = transform(image).unsqueeze(0).to(device)
    
    # 预测
    model.eval()
    with torch.no_grad():
        output = model(image_tensor)
        _, predicted_idx = torch.max(output, 1)
        predicted_class = idx_to_class[predicted_idx.item()]
    
    return predicted_class

In [11]:
# 测试预测
test_image = r'D:\STUDY\internship\raw_data\100µm_10Hz_5N\Séquence images\im0006.tif'  # 替换为实际路径
print(f"测试图像预测: {predict_image(test_image)}")

测试图像预测: two
